In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
data = pd.read_csv('/content/drive/MyDrive/2차 프로젝트/hotel_bookings.csv')
df_raw = data.copy()

In [ ]:
#전처리 없이 돌렸을때(모델돌리기위한 최소한의 전처리만 포함) 결과 확인
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. 정답지와 다름없는(Leakage) 컬럼 리스트 정의
leakage_cols = ['reservation_status', 'reservation_status_date']

# 2. 쌩데이터에서 타겟 변수 및 정답 복사 컬럼 제거
target_col = 'is_canceled'
X_raw = df_raw.drop(columns=[target_col] + leakage_cols, errors='ignore')
y_raw = df_raw[target_col]

# 3. 범주형(문자열) 컬럼 원핫 인코딩
#get_dummies = 판다스의 oneHotEncoding임. NaN데이터도 하나의 컬럼으로 만듦
X_minimal = pd.get_dummies(X_raw, dummy_na=True)

# 4. 남은 수치형 NaN은 중앙값(median)으로 대충 대치
X_minimal = X_minimal.fillna(X_minimal.median())

# 5. Train / Test 분할 (기존 설정 동기화)
X_train_min, X_test_min, y_train_min, y_test_min = train_test_split(
    X_minimal, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# 6. 초고속 RandomForest 가동
base_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
base_model.fit(X_train_min, y_train_min)

# 7. 결과 출력
y_pred_min = base_model.predict(X_test_min)
print(f"\n전처리 전 베이스라인 정확도]: {accuracy_score(y_test_min, y_pred_min):.4f}")
print(classification_report(y_test_min, y_pred_min))


전처리 전 베이스라인 정확도]: 0.7956
              precision    recall  f1-score   support

           0       0.76      0.99      0.86     15033
           1       0.96      0.47      0.63      8845

    accuracy                           0.80     23878
   macro avg       0.86      0.73      0.74     23878
weighted avg       0.83      0.80      0.77     23878



In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
data.shape

(119390, 32)

In [ ]:
data.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [ ]:
data.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='object')

In [ ]:
data.isna().sum()

,0
hotel,0
is_canceled,0
lead_time,0
arrival_date_year,0
arrival_date_month,0
arrival_date_week_number,0
arrival_date_day_of_month,0
stays_in_weekend_nights,0
stays_in_week_nights,0
adults,0


In [ ]:
month_map = {
        'January':'01', 'February':'02', 'March':'03', 'April':'04',
        'May':'05', 'June':'06', 'July':'07', 'August':'08',
        'September':'09', 'October':'10', 'November':'11', 'December':'12'
    }
data['arrival_date_month_num'] = data['arrival_date_month'].map(month_map)

    # 년-월-일 합쳐서 진짜 datetime 객체로 만들기
data['arrival_date'] = pd.to_datetime(
        data['arrival_date_year'].astype(str) + '-' +
        data['arrival_date_month_num'] + '-' +
        data['arrival_date_day_of_month'].astype(str),
        errors='coerce'
  )
    # status date도 날짜형 변환
data['reservation_status_date'] = pd.to_datetime(data['reservation_status_date'], errors='coerce')

In [ ]:
import numpy as np
#파생변수
#예약했던 방이랑 배정받은 방이 다른지?
data['room_assignment_changed'] = (data['reserved_room_type'] != data['assigned_room_type']).astype(int)
#호텔에 머문 주말 일수 + 주중 일수 = 총 일수
data['total_stay_nights'] = data['stays_in_weekend_nights'] + data['stays_in_week_nights']
#아기, 어린이, 성인 다 합친 컬럼 생성 ** Children 컬럼 결측치 4개있음으로 0으로 치환
data['total_people'] = data['adults'] + data['children'].fillna(0) + data['babies']
#그날 방 가격이 0원이었던 방들은 프로모션, 쿠폰 등 다양한 요소 작용 그래서 adr = 0유무 확인
data['is_adr_0'] = (data['adr'] == 0).astype(int)
#Agent가 포함 되어있는 예약인지 확인하는 컬럼 ( 기존 Agent컬럼은 삭제)
data['Agent_check'] = (~data['agent'].isna()).astype(int)
#데이터가 포르투갈 호텔대상으로 한 데이터라 country컬럼에서 포르투갈인일때, 혹은 외국인일때를 분석 할 수 있는 컬럼 생성.
data['foreigner'] = (data['country'] != 'PRT').astype(int)
data.head(10)
#

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date_month_num,arrival_date,room_assignment_changed,total_stay_nights,total_people,is_adr_0,Agent_check,foreigner
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,07,2015-07-01,0,0,2.0,1,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,07,2015-07-01,0,0,2.0,1,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,07,2015-07-01,1,1,1.0,0,0,1
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,07,2015-07-01,0,1,1.0,0,1,1
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,07,2015-07-01,0,2,2.0,0,1,1
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,07,2015-07-01,0,2,2.0,0,1,1
6,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,NaN,NaN,0,Transient,107.0,0,0,Check-Out,2015-07-03,07,2015-07-01,0,2,2.0,0,0,0
7,Resort Hotel,0,9,2015,July,27,1,0,2,2,0.0,0,FB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,303.0,NaN,0,Transient,103.0,0,1,Check-Out,2015-07-03,07,2015-07-01,0,2,2.0,0,1,0
8,Resort Hotel,1,85,2015,July,27,1,0,3,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,82.0,0,1,Canceled,2015-05-06,07,2015-07-01,0,3,2.0,0,1,0
9,Resort Hotel,1,75,2015,July,27,1,0,3,2,0.0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,D,D,0,No Deposit,15.0,NaN,0,Transient,105.5,0,0,Canceled,2015-04-22,07,2015-07-01,0,3,2.0,0,1,0


In [ ]:
import numpy as np
numeric_df = data.select_dtypes(include=[np.number])
numeric_df

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,room_assignment_changed,total_stay_nights,total_people,is_adr_0,Agent_check,foreigner
0,0,342,2015,27,1,0,0,2,0.0,0,0,0,0,3,NaN,NaN,0,0.00,0,0,0,0,2.0,1,0,0
1,0,737,2015,27,1,0,0,2,0.0,0,0,0,0,4,NaN,NaN,0,0.00,0,0,0,0,2.0,1,0,0
2,0,7,2015,27,1,0,1,1,0.0,0,0,0,0,0,NaN,NaN,0,75.00,0,0,1,1,1.0,0,0,1
3,0,13,2015,27,1,0,1,1,0.0,0,0,0,0,0,304.0,NaN,0,75.00,0,0,0,1,1.0,0,1,1
4,0,14,2015,27,1,0,2,2,0.0,0,0,0,0,0,240.0,NaN,0,98.00,0,1,0,2,2.0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,0,23,2017,35,30,2,5,2,0.0,0,0,0,0,0,394.0,NaN,0,96.14,0,0,0,7,2.0,0,1,1
119386,0,102,2017,35,31,2,5,3,0.0,0,0,0,0,0,9.0,NaN,0,225.43,0,2,0,7,3.0,0,1,1
119387,0,34,2017,35,31,2,5,2,0.0,0,0,0,0,0,9.0,NaN,0,157.71,0,4,0,7,2.0,0,1,1
119388,0,109,2017,35,31,2,5,2,0.0,0,0,0,0,0,89.0,NaN,0,104.40,0,0,0,7,2.0,0,1,1


In [ ]:
corr_matrix = numeric_df.corr().abs()
target_corr = corr_matrix['is_canceled'].sort_values(ascending=False)
print(target_corr)

is_canceled                       1.000000
foreigner                         0.336122
lead_time                         0.293123
room_assignment_changed           0.247770
total_of_special_requests         0.234658
required_car_parking_spaces       0.195498
booking_changes                   0.144381
previous_cancellations            0.110133
Agent_check                       0.102068
is_repeated_guest                 0.084793
agent                             0.083114
is_adr_0                          0.069990
adults                            0.060017
previous_bookings_not_canceled    0.057358
days_in_waiting_list              0.054186
adr                               0.047557
total_people                      0.046522
babies                            0.032491
stays_in_week_nights              0.024765
company                           0.020642
total_stay_nights                 0.017779
arrival_date_year                 0.016660
arrival_date_week_number          0.008148
arrival_dat

In [ ]:
drop_cols = [
        'reservation_status',          # 데이터 누수
        'arrival_date_year',           # 연도 과적합 방지
        'arrival_date_week_number',    # 중복 정보
        'arrival_date_month_num',      # 임시 컬럼
        'agent', 'company',
        'reservation_status_date',  # 유무만 남기고 원본 ID 컬럼은 삭제
        'country'
    ]

In [ ]:
data = data.drop(columns=drop_cols, errors='ignore')

In [ ]:

data.shape

(119390, 32)

In [ ]:
data.head()

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,arrival_date,room_assignment_changed,total_stay_nights,total_people,is_adr_0,Agent_check,foreigner
0,Resort Hotel,0,342,July,1,0,0,2,0.0,0,BB,Direct,Direct,0,0,0,C,C,3,No Deposit,0,Transient,0.0,0,0,2015-07-01,0,0,2.0,1,0,0
1,Resort Hotel,0,737,July,1,0,0,2,0.0,0,BB,Direct,Direct,0,0,0,C,C,4,No Deposit,0,Transient,0.0,0,0,2015-07-01,0,0,2.0,1,0,0
2,Resort Hotel,0,7,July,1,0,1,1,0.0,0,BB,Direct,Direct,0,0,0,A,C,0,No Deposit,0,Transient,75.0,0,0,2015-07-01,1,1,1.0,0,0,1
3,Resort Hotel,0,13,July,1,0,1,1,0.0,0,BB,Corporate,Corporate,0,0,0,A,A,0,No Deposit,0,Transient,75.0,0,0,2015-07-01,0,1,1.0,0,1,1
4,Resort Hotel,0,14,July,1,0,2,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,0,Transient,98.0,0,1,2015-07-01,0,2,2.0,0,1,1


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [ ]:
#  타겟 분리
X = data.drop('is_canceled', axis=1)
y = data['is_canceled']

In [ ]:
data.head(1)

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,arrival_date,room_assignment_changed,total_stay_nights,total_people,is_adr_0,Agent_check,foreigner
0,Resort Hotel,0,342,July,1,0,0,2,0.0,0,BB,Direct,Direct,0,0,0,C,C,3,No Deposit,0,Transient,0.0,0,0,2015-07-01,0,0,2.0,1,0,0


In [ ]:
# 데이터 분리 (전처리 전에 무조건 먼저)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:

print(f'최대값: {data['adr'].max()} 평균: {data['adr'].mean()}')
print(f'최대값: {data['total_people'].max()} 평균: {data['total_people'].mean()}')
print(f'최대값: {data['total_stay_nights'].max()} 평균: {data['total_stay_nights'].mean()}')

최대값: 5400.0 평균: 101.83112153446686
최대값: 55.0 평균: 1.9682385459418712
최대값: 69 평균: 3.4279001591423066


In [ ]:
import pandas as pd

total_initial_rows = len(X_train)
all_outlier_indices = set()

# 1. adr(객실 가격)은 IQR 기준으로 극단적 이상치(whisker=3.0)만 저격
# (가끔 0원 뻥튀기나 수천 달러짜리 비정상 데이터만 컷)
col = 'adr'
Q1, Q3 = X_train[col].quantile(0.25), X_train[col].quantile(0.75)
IQR = Q3 - Q1
adr_outliers = X_train[(X_train[col] < Q1 - 3.0 * IQR) | (X_train[col] > Q3 + 3.0 * IQR)]
all_outlier_indices.update(adr_outliers.index)

# 2. total_people(총 인원)은 IQR을 쓰지 않고 상식적인 도메인 기준으로 컷!
# 일반 객실에 '10명 초과'로 예약된 건 명백한 데이터 입력 오류나 특이 단체 건이므로 이것만 저격
if 'total_people' in X_train.columns:
    people_outliers = X_train[X_train['total_people'] > 10]
    all_outlier_indices.update(people_outliers.index)

# 3. 드롭 및 y_train 싱크 맞추기
X_train = X_train.drop(index=list(all_outlier_indices), errors='ignore')
y_train = y_train.loc[X_train.index]

In [ ]:
X_train.shape, y_train.shape

((95239, 31), (95239,))

In [ ]:
# 숫자형 / 범주형 컬럼 이름 추출 (X_train 기준)
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# 파이프라인 구축

#숫자컬럼 - 결축지: 중앙값, 엔코딩:StandardScaler
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
#문자컬럼 - 결축지: 최빈값, 엔코딩:OneHotEncoder
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # sparse_output=False 추가
])

#최종 전처리기 조립
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
!pip install xgboost # xgboost는 google colab에서 지원 안해서 설치해야함
from xgboost import XGBClassifier


In [ ]:
models = {
    # Logistic Regression은 피처 스케일에 민감해서 StandardScaler 필수
    # Pipeline으로 묶으면 학습/예측 시 스케일링이 자동으로 적용됨
    "Logistic Regression":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                random_state=42
            ))
        ]),
    # 트리 계열 모델은 스케일링 불필요
    "Decision Tree":
        DecisionTreeClassifier(
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        )
}


In [ ]:
# 8. Base 모델 학습 및 평가

results = []

best_model_name = ""
best_auc = 0  # 최고 모델 선정 기준을 AUC로 변경
              # Accuracy는 불균형 데이터(유지 63% / 취소 37%)에서 신뢰하기 어려움
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

for name, model in models.items():

    print("\n" + "="*60)
    print(f"{name} 학습 시작")
    print("="*60)
    model.fit(
        X_train_processed,
        y_train
    )
    pred  = model.predict(X_test_processed)
    proba = model.predict_proba(X_test_processed)[:, 1]  # 취소(1)일 확률 — AUC 계산에 사용

    acc       = accuracy_score(y_test, pred)
    recall    = recall_score(y_test, pred)
    precision = precision_score(y_test, pred)
    auc       = roc_auc_score(y_test, proba)
    results.append({
        "Model":     name,
        "AUC":       round(auc, 4),
        "Accuracy":  round(acc, 4),
        "Recall":    round(recall, 4),
        "Precision": round(precision, 4)
    })
    print(classification_report(y_test, pred))
    print(f"AUC: {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        best_model_name = name



Logistic Regression 학습 시작
              precision    recall  f1-score   support

           0       0.82      0.91      0.86     15033
           1       0.81      0.65      0.72      8845

    accuracy                           0.81     23878
   macro avg       0.81      0.78      0.79     23878
weighted avg       0.81      0.81      0.81     23878

AUC: 0.8928

Decision Tree 학습 시작
              precision    recall  f1-score   support

           0       0.88      0.87      0.88     15033
           1       0.79      0.80      0.79      8845

    accuracy                           0.85     23878
   macro avg       0.83      0.84      0.84     23878
weighted avg       0.85      0.85      0.85     23878

AUC: 0.8393

Random Forest 학습 시작
              precision    recall  f1-score   support

           0       0.89      0.93      0.91     15033
           1       0.88      0.80      0.84      8845

    accuracy                           0.88     23878
   macro avg       0.88      0.87  

In [ ]:
# 9. 결과 정리

result_df = pd.DataFrame(results)
# AUC 기준으로 정렬
result_df = result_df.sort_values(
    by="AUC",
    ascending=False
)
print("\n===== 모델 비교 결과 =====")
print(result_df)



===== 모델 비교 결과 =====
                 Model     AUC  Accuracy  Recall  Precision
2        Random Forest  0.9513    0.8845  0.8002     0.8771
4              XGBoost  0.9437    0.8701  0.7822     0.8547
3    Gradient Boosting  0.9193    0.8445  0.7401     0.8223
0  Logistic Regression  0.8928    0.8140  0.6488     0.8112
1        Decision Tree  0.8393    0.8463  0.8009     0.7876


In [ ]:
from sklearn.model_selection import GridSearchCV
# 튜닝 대상 모델 2개 선정

tuning_models = {
    "Random Forest": {
        "model": RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [10, 20],
            "min_samples_leaf": [5, 10]
        }
    },

    "XGBoost": {
        "model": XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1]
        }
    }
}

In [ ]:
# GridSearchCV로 하이퍼파라미터 튜닝

tuning_results = []

for name, info in tuning_models.items():
    print(f"\n===== {name} 튜닝 시작 =====")

    grid = GridSearchCV(
        estimator=info["model"],
        param_grid=info["params"],
        scoring="roc_auc",
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train_processed, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test_processed)
    y_proba = best_model.predict_proba(X_test_processed)[:, 1]

    tuning_results.append({
        "model": name,
        "best_estimator": best_model,
        "best_cv_auc": grid.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_auc": roc_auc_score(y_test, y_proba),
        "best_params": grid.best_params_
    })

    print(f"{name} 완료")
    print("Best Params:", grid.best_params_)
    print("Best CV AUC:", grid.best_score_)


===== Random Forest 튜닝 시작 =====
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Random Forest 완료
Best Params: {'max_depth': 20, 'min_samples_leaf': 5, 'n_estimators': 200}
Best CV AUC: 0.9382358242221573

===== XGBoost 튜닝 시작 =====
Fitting 3 folds for each of 8 candidates, totalling 24 fits
XGBoost 완료
Best Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best CV AUC: 0.9353227503563147


In [ ]:
tuning_result_df = pd.DataFrame(tuning_results)

tuning_result_df = tuning_result_df.sort_values(
    by="test_auc",
    ascending=False
)

tuning_result_df

,model,best_estimator,best_cv_auc,test_accuracy,test_precision,test_recall,test_auc,best_params
0,Random Forest,"(DecisionTreeClassifier(max_depth=20, max_feat...",0.938236,0.862970,0.873876,0.736348,0.941123,"{'max_depth': 20, 'min_samples_leaf': 5, 'n_es..."
1,XGBoost,"XGBClassifier(base_score=None, booster=None, c...",0.935323,0.859201,0.845408,0.758621,0.935188,"{'learning_rate': 0.1, 'max_depth': 5, 'n_esti..."


In [ ]:
grid.best_estimator_

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

In [ ]:
#  전처리기 + 모델을 하나로 합친 최종 통합 파이프라인
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', grid.best_estimator_)
    ])

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['lead_time', 'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [ ]:
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print(f"최종 AUC: {auc:.4f}")


최종 AUC: 0.9352


In [ ]:
# 3. 파일 저장
import joblib
joblib.dump(model, '/content/drive/MyDrive/2차 프로젝트/best_model.pkl')

['/content/drive/MyDrive/2차 프로젝트/best_model.pkl']